# Clustering Consulting Project - Solutions

#### Task
A large technology firm needs your help: they were hacked. The forensic team collected metadata from each suspicious session (session time, transfer size, typing speed, etc.). The available features are:

- `Session_Connection_Time`: session duration in minutes
- `Bytes Transferred`: MB transferred during the session
- `Kali_Trace_Used`: whether Kali Linux was used
- `Servers_Corrupted`: number of corrupted servers
- `Pages_Corrupted`: number of illegally accessed pages
- `Location`: source location (likely low value because attackers used VPNs)
- `WPM_Typing_Speed`: estimated typing speed from logs

There are 3 suspects. The firm is confident about the first two, but unsure about the third. The goal is to use clustering to estimate whether the attacks were likely done by 2 or 3 hackers.

A key clue from the forensic engineer: attackers usually trade off, so attack counts per hacker should be roughly balanced. For example, with 100 attacks: around 50/50 for 2 hackers, or around 33/33/34 for 3 hackers.

In [0]:
from collections import Counter
from pyspark.sql import SparkSession
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from contextlib import redirect_stderr
import io

#### Start Spark session and load the dataset.

In [0]:
spark = SparkSession.builder.appName('hack_clustering').getOrCreate()
attack_logs_df = spark.read.csv('data/hack_data.csv', header=True, inferSchema=True)

In [0]:
attack_logs_df.head()

Row(Session_Connection_Time=8.0, Bytes Transferred=391.09, Kali_Trace_Used=1, Servers_Corrupted=2.96, Pages_Corrupted=7.0, Location='Slovenia', WPM_Typing_Speed=72.37)

In [0]:
attack_logs_df.describe().show()

+-------+-----------------------+-----------------+------------------+------------------+------------------+-----------+------------------+
|summary|Session_Connection_Time|Bytes Transferred|   Kali_Trace_Used| Servers_Corrupted|   Pages_Corrupted|   Location|  WPM_Typing_Speed|
+-------+-----------------------+-----------------+------------------+------------------+------------------+-----------+------------------+
|  count|                    334|              334|               334|               334|               334|        334|               334|
|   mean|     30.008982035928145|607.2452694610777|0.5119760479041916| 5.258502994011977|10.838323353293413|       NULL|57.342395209580864|
| stddev|     14.088200614636174|286.3359316357677|0.5006065264451391|2.3019069333969684| 3.063526330360217|       NULL|13.411063368434624|
|    min|                    1.0|             10.0|                 0|               1.0|               6.0|Afghanistan|              40.0|
|    max|           

In [0]:
attack_logs_df.columns

['Session_Connection_Time',
 'Bytes Transferred',
 'Kali_Trace_Used',
 'Servers_Corrupted',
 'Pages_Corrupted',
 'Location',
 'WPM_Typing_Speed']

In [0]:
attack_logs_df.printSchema()

root
 |-- Session_Connection_Time: double (nullable = true)
 |-- Bytes Transferred: double (nullable = true)
 |-- Kali_Trace_Used: integer (nullable = true)
 |-- Servers_Corrupted: double (nullable = true)
 |-- Pages_Corrupted: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- WPM_Typing_Speed: double (nullable = true)



In [0]:
feature_columns = [
    'Session_Connection_Time',
    'Bytes Transferred',
    'Kali_Trace_Used',
    'Servers_Corrupted',
    'Pages_Corrupted',
    'WPM_Typing_Speed',
]

spark_features_df = attack_logs_df.select(*feature_columns)
spark_features_df.show(3, truncate=False)

+-----------------------+-----------------+---------------+-----------------+---------------+----------------+
|Session_Connection_Time|Bytes Transferred|Kali_Trace_Used|Servers_Corrupted|Pages_Corrupted|WPM_Typing_Speed|
+-----------------------+-----------------+---------------+-----------------+---------------+----------------+
|8.0                    |391.09           |1              |2.96             |7.0            |72.37           |
|20.0                   |720.99           |0              |3.04             |9.0            |69.08           |
|31.0                   |356.32           |1              |3.71             |8.0            |70.58           |
+-----------------------+-----------------+---------------+-----------------+---------------+----------------+
only showing top 3 rows


#### Keep feature processing in Spark, then standardize features with sklearn.

In [0]:
pandas_features_df = spark_features_df.toPandas()

scaler = StandardScaler()
scaled_features = scaler.fit_transform(pandas_features_df)

#### Compare sklearn KMeans with K=2 vs K=3.

In [0]:
stderr_buffer = io.StringIO()

with redirect_stderr(stderr_buffer):
    kmeans_three_model = KMeans(n_clusters=3, random_state=42, n_init=20)
    kmeans_two_model = KMeans(n_clusters=2, random_state=42, n_init=20)

    labels_three = kmeans_three_model.fit_predict(scaled_features)
    labels_two = kmeans_two_model.fit_predict(scaled_features)

    wssse_three = kmeans_three_model.inertia_
    wssse_two = kmeans_two_model.inertia_

print('With K=3')
print('Within Set Sum of Squared Errors = ' + str(wssse_three))
print('With K=2')
print('Within Set Sum of Squared Errors = ' + str(wssse_two))

With K=3
Within Set Sum of Squared Errors = 435.4530414928184
With K=2
Within Set Sum of Squared Errors = 603.577870640845


#### WSSSE naturally decreases as K increases, so inspect the trend across multiple K values.

In [0]:
stderr_buffer = io.StringIO()

with redirect_stderr(stderr_buffer):
    for cluster_count in range(2, 9):
        kmeans_model = KMeans(n_clusters=cluster_count, random_state=42, n_init=20)
        kmeans_model.fit(scaled_features)
        current_wssse = kmeans_model.inertia_
        print(f'With K={cluster_count}')
        print('Within Set Sum of Squared Errors = ' + str(current_wssse))

With K=2
Within Set Sum of Squared Errors = 603.577870640845
With K=3
Within Set Sum of Squared Errors = 435.4530414928184
With K=4
Within Set Sum of Squared Errors = 267.9358147268938
With K=5
Within Set Sum of Squared Errors = 246.1010447728541
With K=6
Within Set Sum of Squared Errors = 225.49988191538864
With K=7
Within Set Sum of Squared Errors = 206.13964710236712
With K=8
Within Set Sum of Squared Errors = 190.85384544528273


#### Use cluster sizes as the key decision signal: if attackers split work evenly, the correct K should produce more balanced group counts.

In [0]:
counts_three = sorted((int(k), int(v)) for k, v in Counter(labels_three).items())
spark.createDataFrame(counts_three, ['prediction', 'count']).orderBy('prediction').show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0|   83|
|         1|  167|
|         2|   84|
+----------+-----+



In [0]:
counts_two = sorted((int(k), int(v)) for k, v in Counter(labels_two).items())
spark.createDataFrame(counts_two, ['prediction', 'count']).orderBy('prediction').show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0|  167|
|         1|  167|
+----------+-----+



#### Conclusion

The evidence points to **2 hackers**: with `K=2`, clustering forms two similarly sized groups, matching the forensic assumption that attackers split activity roughly evenly.